# 3. Threshold Optimization & Analysis

Analyze different decision thresholds and find optimal threshold for loan default prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## 3.1 Load Data & Train Model

In [ ]:
# Load and preprocess
df = pd.read_csv('../data/loans.csv')
X = df.drop(['LoanID', 'Default'], axis=1)
y = df['Default']

# Encode
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Get probability predictions
y_test_pred_proba = model.predict_proba(X_test)[:, 1]

print(f"Model trained. Predictions ready: {len(y_test_pred_proba)} samples")

## 3.2 Test Multiple Thresholds

In [ ]:
# Test thresholds
thresholds = [0.5, 0.3, 0.2, 0.15, 0.1]
results = []

for threshold in thresholds:
    y_pred = (y_test_pred_proba >= threshold).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Flagged': (y_pred == 1).sum()
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

## 3.3 Visualize Threshold Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy vs Threshold
axes[0, 0].plot(results_df['Threshold'], results_df['Accuracy'], marker='o', linewidth=2.5, markersize=8)
axes[0, 0].set_title('Accuracy vs Threshold', fontweight='bold')
axes[0, 0].set_xlabel('Threshold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].grid(alpha=0.3)

# Precision vs Recall
axes[0, 1].plot(results_df['Recall'], results_df['Precision'], marker='s', linewidth=2.5, markersize=8)
axes[0, 1].set_title('Precision vs Recall Tradeoff', fontweight='bold')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].grid(alpha=0.3)

# Recall vs Threshold
axes[1, 0].plot(results_df['Threshold'], results_df['Recall'], marker='^', linewidth=2.5, markersize=8)
axes[1, 0].set_title('Recall vs Threshold', fontweight='bold')
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].grid(alpha=0.3)

# F1-Score vs Threshold
axes[1, 1].plot(results_df['Threshold'], results_df['F1-Score'], marker='d', linewidth=2.5, markersize=8)
axes[1, 1].set_title('F1-Score vs Threshold', fontweight='bold')
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Recommended Threshold: 0.20 (balances recall and precision)")

## 3.4 Business Impact Analysis

In [ ]:
print("="*80)
print("BUSINESS IMPACT ANALYSIS @ THRESHOLD 0.20")
print("="*80)

threshold = 0.20
y_pred = (y_test_pred_proba >= threshold).astype(int)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

print(f"\nTrue Negatives (TN):   {tn:>6,}  (Correctly Approved)")
print(f"False Positives (FP):  {fp:>6,}  (Flagged for Review)")
print(f"False Negatives (FN):  {fn:>6,}  (Missed Defaults)")
print(f"True Positives (TP):   {tp:>6,}  (Caught Defaults)")

print(f"\nCost Analysis (assuming $50K loss per default, $50 review cost):")
review_cost = fp * 50
missed_default_cost = fn * 50000

print(f"  Review Cost (FP):        ${review_cost:>15,}")
print(f"  Missed Default Cost (FN): ${missed_default_cost:>15,}")
print(f"  Total Cost:              ${review_cost + missed_default_cost:>15,}")